<a href="https://colab.research.google.com/github/Midunvarsan/Codtech-Intern/blob/main/Neural_Style_Transfer_Explanation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from PIL import Image
import gradio as gr
import io
import os

# Set TensorFlow logging to suppress warnings during model loading
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- 1. CORE FUNCTIONALITY & MODEL ---

# Load the pre-trained Neural Style Transfer model from TensorFlow Hub
# This model implements arbitrary image stylization using a feed-forward network,
# making it fast and suitable for interactive applications.
# The chosen model is 'magenta/arbitrary-image-stylization-v1-256'
# which is known for its speed and quality.
print("Loading Neural Style Transfer model from TensorFlow Hub...")
st_model = hub.load('https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2')
print("Model loaded successfully!")

def preprocess_image(image: Image.Image, target_dim: int = 512, is_style: bool = False) -> tf.Tensor:
    """
    Preprocesses an input image for the Neural Style Transfer model.

    Args:
        image (PIL.Image.Image): The input image (content or style) as a PIL Image object.
        target_dim (int): The target dimension for the larger side of the image after resizing.
                          For content images, this helps manage memory and computational load.
                          For style images, the model internally resizes to 256x256, but initial
                          resizing here ensures consistent input handling.
        is_style (bool): Flag to indicate if the image is a style image. Style images are
                         typically resized differently or have different preprocessing nuances.

    Returns:
        tf.Tensor: A preprocessed TensorFlow tensor, ready for model input.
                   The tensor will be of shape (1, height, width, 3) and pixel values
                   will be normalized to the range [0, 1].
    """
    # Convert PIL Image to Bytes for TensorFlow decoding (if not already a bytes object)
    if isinstance(image, Image.Image):
        img_byte_arr = io.BytesIO()
        image.save(img_byte_arr, format='PNG') # Use PNG to avoid compression artifacts
        img_bytes = img_byte_arr.getvalue()
    else:
        raise ValueError("Input image must be a PIL.Image.Image object.")

    # Decode the image bytes to a TensorFlow tensor
    img_tensor = tf.image.decode_image(img_bytes, channels=3, dtype=tf.float32)
    # Add a batch dimension: (height, width, 3) -> (1, height, width, 3)
    img_tensor = tf.expand_dims(img_tensor, axis=0)

    # Normalize image pixels to the range [0, 1]
    # The TensorFlow Hub model expects input images in this range.
    if tf.reduce_max(img_tensor) > 1.0:
        img_tensor = img_tensor / 255.0

    # Resize the image while maintaining aspect ratio
    original_shape = tf.shape(img_tensor)[1:3] # (height, width)
    height, width = original_shape[0], original_shape[1]

    if height > width:
        scale = tf.cast(target_dim / height, tf.float32)
        new_height = target_dim
        new_width = tf.cast(scale * width, tf.int32)
    else:
        scale = tf.cast(target_dim / width, tf.float32)
        new_width = target_dim
        new_height = tf.cast(scale * height, tf.int32)

    # For style images, the model itself will internally resize to 256x256.
    # However, pre-resizing helps in some cases for consistent input handling
    # or if the model's internal resizing is sensitive to very large inputs.
    # For content images, resizing helps reduce computation.
    if is_style:
        # Style images are typically resized to a fixed square for feature extraction.
        # The model usually handles this internally, but for consistency, we can resize them here too.
        # Using 256 as a common dimension, as the model description mentions 256x256.
        img_tensor = tf.image.resize(img_tensor, [256, 256], method=tf.image.ResizeMethod.LANCZOS3)
    else:
        # Content images are resized to a maximum dimension while preserving aspect ratio.
        # This prevents extremely large images from consuming too much memory/time.
        img_tensor = tf.image.resize(img_tensor, [new_height, new_width], method=tf.image.ResizeMethod.LANCZOS3)

    # Ensure the shape is (1, H, W, 3)
    img_tensor.set_shape([1, None, None, 3])

    return img_tensor

def apply_style_transfer(content_img_pil: Image.Image, style_img_pil: Image.Image) -> Image.Image:
    """
    Applies neural style transfer from a style image to a content image.

    Args:
        content_img_pil (PIL.Image.Image): The content image as a PIL Image object.
        style_img_pil (PIL.Image.Image): The style image as a PIL Image object.

    Returns:
        PIL.Image.Image: The stylized image as a PIL Image object.

    Raises:
        Exception: If an error occurs during style transfer (e.g., model inference issue).
    """
    try:
        # Preprocess content and style images
        # Content image target_dim can be adjusted for higher resolution output, but impacts speed/memory.
        # 512 is a good balance for interactive demos.
        content_image_tensor = preprocess_image(content_img_pil, target_dim=512, is_style=False)
        # Style image is preprocessed and then the style bottleneck features are extracted.
        # The model expects style images to be roughly 256x256, handled by preprocess_image's is_style flag.
        style_image_tensor = preprocess_image(style_img_pil, target_dim=256, is_style=True)

        # Extract style bottleneck features from the style image
        # These features encode the artistic style characteristics.
        style_bottleneck_features = st_model(tf.constant(content_image_tensor), tf.constant(style_image_tensor))[1]

        # Apply style transfer using the content image and extracted style features
        # The model generates the stylized image directly from the inputs.
        stylized_image_tensor = st_model(tf.constant(content_image_tensor), style_bottleneck_features)[0]

        # Post-process the output tensor to a PIL Image
        # Remove batch dimension: (1, H, W, 3) -> (H, W, 3)
        stylized_image_np = stylized_image_tensor[0].numpy()
        # Denormalize pixel values from [0, 1] to [0, 255] and convert to uint8
        stylized_image_np = (stylized_image_np * 255).astype(np.uint8)
        # Convert NumPy array to PIL Image
        stylized_img_pil = Image.fromarray(stylized_image_np)

        return stylized_img_pil

    except Exception as e:
        print(f"Error during style transfer: {e}")
        raise

# --- 2. USER INTERFACE (UI) --- #

def launch_interface():
    """
    Launches the Gradio web interface for the Neural Style Transfer application.
    The UI allows users to upload content and style images, then generate
    and display the stylized output along with the inputs.
    """
    iface = gr.Interface(
        fn=apply_style_transfer,
        inputs=[
            gr.Image(type="pil", label="Upload Content Image"),
            gr.Image(type="pil", label="Upload Style Image")
        ],
        outputs=gr.Image(type="pil", label="Stylized Output Image"),
        title="🖼️ Neural Style Transfer with Gradio 🎨",
        description="""
        <h3>Welcome to the Neural Style Transfer App!</h3>
        <p>This application allows you to transfer the artistic style from one image (Style Image)
        to another image (Content Image). Powered by a fast TensorFlow Hub model.</p>
        <p><b>Instructions:</b></p>
        <ol>
            <li>Upload your 'Content Image' (e.g., a photograph).</li>
            <li>Upload your 'Style Image' (e.g., a painting, abstract art).</li>
            <li>Click the 'Submit' button to generate your stylized image.</li>
        </ol>
        <p>This project is part of an AI internship with CODTECH IT SOLUTIONS.</p>
        """,
        # Removed 'examples' to prevent download errors from inaccessible URLs
        allow_flagging="never", # Disable data flagging for this portfolio project
        live=False # Set to False for explicit button press
    )

    print("Launching Gradio interface...")
    # Launch the interface. In Colab, this will provide a public URL.
    iface.launch(debug=True, share=True)

# --- Main Execution --- #
if __name__ == "__main__":
    launch_interface()